In [1]:
import sys

import torch

import pandas as pd

from config.feature_config import FeatureConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from dice4el.scenario.scenario_handler import ScenarioHandler

from dice4el.scenario.scenario_model import ScenarioLSTM
from dice4el.scenario.scenario_model_wrapper import ScenarioModelWrapper

from dice4el.dice4el_config import EventLogDiCEConfig
from dice4el.eventlog_dice import EventLogDiCE
from dice4el.eventlog_dice_optimized import EventLogDiCEOptimized

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [2]:
set_seed(seed=777)

In [3]:
df = pd.read_excel(
    "../../../../data/bpic20_Ptc.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "org:resource": "string",
        "org:role": "string",
        "case:Activity": "string",
        "case:OrganizationalEntity": "string",
        "case:RequestedAmount": "float32",
        "case:Permit RequestedBudget": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [4]:
df.head(20)

,case:concept:name,time:timestamp,case:Activity,case:OrganizationalEntity,case:Permit OrganizationalEntity,case:Permit RequestedBudget,case:RequestedAmount,concept:name,org:resource,org:role,time_delta
0,request for payment 1000,2018-03-01 10:55:17,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Permit SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
1,request for payment 1000,2018-03-01 10:55:21,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Permit APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,4.0
2,request for payment 1000,2018-03-01 11:34:16,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Request For Payment SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,2335.0
3,request for payment 1000,2018-03-01 11:34:23,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Request For Payment APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,7.0
4,request for payment 1000,2018-03-01 15:01:48,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Permit FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,12445.0
5,request for payment 1000,2018-03-05 14:49:53,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Request For Payment FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,344885.0
6,request for payment 1000,2018-03-06 10:13:29,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Request Payment,SYSTEM,UNDEFINED,69816.0
7,request for payment 1000,2018-03-08 17:31:00,UNKNOWN,organizational unit 65454,organizational unit 65460,1273.252075,365.177460,Payment Handled,SYSTEM,UNDEFINED,199051.0
8,request for payment 10043,2018-02-20 13:53:11,activity 505,organizational unit 65468,organizational unit 65466,2531.512695,2129.845947,Permit SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
9,request for payment 10043,2018-02-20 13:53:14,activity 505,organizational unit 65468,organizational unit 65466,2531.512695,2129.845947,Permit APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,3.0


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
feature_config = FeatureConfig.load(
    path = "../../pretrained_models/"
)

In [7]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:Activity', 'case:OrganizationalEntity', 'case:Permit RequestedBudget', 'case:RequestedAmount', 'concept:name', 'org:resource', 'org:role', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [6.00, 362277.00]                        61266.0000 quantile_derived    
case:RequestedAmount           continuous     case     yes    [64.68, 1661.05]                         338.4588   quantile_derived    
case:Permit RequestedBudget    continuous     case     yes    [130.85, 4066.04]                        769

In [8]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../../pretrained_models/"
)

In [9]:
model = ProcessLSTM.load(
    path = "../../pretrained_models/"
)

In [10]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

In [11]:
scenario_handler =  ScenarioHandler(
    feature_config=feature_config,
    preprocessor_artifacts=preprocessor_artifacts
)

In [12]:
scenario_model = ScenarioLSTM.load(
    path = "../pretrained_models/"
)

In [13]:
scenario_model_wrapper = ScenarioModelWrapper(
    scenario_model=scenario_model,
    scenario_handler=scenario_handler,
    device=device
)

### --- Process Constraints ---

In [14]:
engine = ProcessModelConstraintEngine.load(
     path = "../../pretrained_models/"
)

In [15]:
engine.parallel_sets

[{'Payment Handled', 'Request Payment'}]

In [16]:
engine.branching_sets

[{'Payment Handled',
  'Permit APPROVED by ADMINISTRATION',
  'Permit APPROVED by BUDGET OWNER',
  'Permit APPROVED by PRE_APPROVER',
  'Permit APPROVED by SUPERVISOR',
  'Permit FINAL_APPROVED by DIRECTOR',
  'Permit FINAL_APPROVED by SUPERVISOR',
  'Permit REJECTED by ADMINISTRATION',
  'Permit REJECTED by BUDGET OWNER',
  'Permit REJECTED by EMPLOYEE',
  'Permit REJECTED by PRE_APPROVER',
  'Permit REJECTED by SUPERVISOR',
  'Permit SUBMITTED by EMPLOYEE',
  'Request For Payment APPROVED by ADMINISTRATION',
  'Request For Payment APPROVED by BUDGET OWNER',
  'Request For Payment APPROVED by PRE_APPROVER',
  'Request For Payment APPROVED by SUPERVISOR',
  'Request For Payment FINAL_APPROVED by DIRECTOR',
  'Request For Payment FINAL_APPROVED by SUPERVISOR',
  'Request For Payment REJECTED by ADMINISTRATION',
  'Request For Payment REJECTED by BUDGET OWNER',
  'Request For Payment REJECTED by EMPLOYEE',
  'Request For Payment REJECTED by PRE_APPROVER',
  'Request For Payment REJECTED 

### --- Load Experiments ---

In [17]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic20_Ptc-cf_seed777_experiments_dice4el_output.txt", console=False)

In [18]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [19]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

### --- Counterfactuals ---

In [20]:
dice4el_config = EventLogDiCEConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=1.0,
    w_margin_loss=1.0,
    w_scenario_loss=1.0,
    w_distance_loss=1.0,
    w_cat_loss=1.0,
)
dice4el_config.validate()

In [21]:
cf_DiCE4EL = EventLogDiCE(
    dice4el_config=dice4el_config,
    next_event_model_wrapper=model_wrapper,
    scenario_model_wrapper=scenario_model_wrapper,
)

results = generator.run_experiment_df(
    cf_method=cf_DiCE4EL,
    technique="DiCE4EL_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/190 [00:00<?, ?case/s]

In [22]:
results

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,request for payment 70592,4,1,0,0.010379,0.020758,0.000000,0.4000,0.000000,...,0.526782,0.000000,0.010379,0.000000,0.020758,0.4000,0.116403,0.116403,0.0,0.000000
1,0,request for payment 29593,4,1,0,0.046766,0.093531,0.000000,0.4000,0.181818,...,1.114211,0.181818,0.046766,0.000000,0.093531,0.4000,0.485627,0.485627,0.0,0.000000
2,0,request for payment 52969,6,1,0,0.040119,0.080239,0.000000,0.4000,0.333333,...,1.647630,0.333333,0.040119,0.000000,0.080239,0.4000,0.874178,0.874178,1.0,1.000000
3,0,request for payment 55558,7,1,0,0.083382,0.000097,0.166667,0.5000,0.470588,...,1.053970,0.470588,0.083382,0.166667,0.000097,0.5000,0.000000,0.000000,0.0,0.000000
4,0,request for payment 45517,8,1,2,0.011655,0.023309,0.000000,0.4000,0.526316,...,1.401850,0.526316,0.011655,0.000000,0.023309,0.4000,0.463880,0.463880,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,18,request for payment 77073,8,1,2,0.000322,0.000643,0.000000,0.3750,0.000000,...,0.375322,0.000000,0.000322,0.000000,0.000643,0.3750,0.000000,0.000000,0.0,0.000000
162,18,request for payment 71978,8,1,2,0.083563,0.167125,0.000000,0.3750,0.000000,...,0.458563,0.000000,0.083563,0.000000,0.167125,0.3750,0.000000,0.000000,0.0,0.000000
163,18,request for payment 82927,8,1,2,0.161942,0.223884,0.100000,0.4375,0.473684,...,2.031960,0.473684,0.161942,0.100000,0.223884,0.4375,0.958834,0.628974,1.0,0.999998
164,18,request for payment 82310,8,1,2,0.030460,0.060921,0.000000,0.3750,0.157895,...,1.507524,0.157895,0.030460,0.000000,0.060921,0.3750,0.944169,0.000000,1.0,0.000000


In [23]:
cf_DiCE4EL_optim = EventLogDiCEOptimized(
    dice4el_config=dice4el_config,
    next_event_model_wrapper=model_wrapper,
    scenario_model_wrapper=scenario_model_wrapper,
)

results_optim = generator.run_experiment_df(
    cf_method=cf_DiCE4EL_optim,
    technique="DiCE4EL-Optimized_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/190 [00:00<?, ?case/s]

In [24]:
results_optim

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,request for payment 70592,4,1,0,0.013904,0.027807,0.000000,0.400,0.000000,...,0.520381,0.000000,0.013904,0.000000,0.027807,0.400,0.106478,0.106478,0.0,0.0
1,0,request for payment 29593,4,1,0,0.179046,0.191426,0.166667,0.500,0.181818,...,1.058561,0.181818,0.179046,0.166667,0.191426,0.500,0.197696,0.197696,0.0,0.0
2,0,request for payment 52969,6,1,0,0.173781,0.180896,0.166667,0.500,0.400000,...,1.560586,0.400000,0.173781,0.166667,0.180896,0.500,0.486805,0.486805,0.0,0.0
3,0,request for payment 55558,7,1,0,0.270966,0.208598,0.333333,0.600,0.470588,...,1.341554,0.470588,0.270966,0.333333,0.208598,0.600,0.000000,0.000000,0.0,0.0
4,0,request for payment 45517,8,1,2,0.057214,0.114427,0.000000,0.400,0.526316,...,1.403728,0.526316,0.057214,0.000000,0.114427,0.400,0.420199,0.420199,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,18,request for payment 77073,8,1,2,0.009266,0.018532,0.000000,0.375,0.000000,...,0.384266,0.000000,0.009266,0.000000,0.018532,0.375,0.000000,0.000000,0.0,0.0
162,18,request for payment 71978,8,1,2,0.086155,0.172310,0.000000,0.375,0.000000,...,0.461155,0.000000,0.086155,0.000000,0.172310,0.375,0.000000,0.000000,0.0,0.0
163,18,request for payment 82927,8,1,2,0.308807,0.217614,0.400000,0.625,0.684211,...,2.559187,0.684211,0.308807,0.400000,0.217614,0.625,0.941170,0.000000,1.0,0.0
164,18,request for payment 82310,8,1,2,0.035068,0.070136,0.000000,0.375,0.157895,...,1.512091,0.157895,0.035068,0.000000,0.070136,0.375,0.944129,0.000000,1.0,0.0


### --- Cleanup ---

In [25]:
sys.stdout = original_stdout
log_file.close()